In [ ]:
import os
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import anndata as ad
from pathlib import Path

In [ ]:
data_dir = Path("/home/anilprakash/labs/Mei/projects/anil/srda/notebooks/data/scrna_seq/kang/")
data_dir.mkdir(parents=True, exist_ok=True)

nhood_size = 50

#specify the condition column name in the adata.obs dataframe

condition_col = 'label'

celltype_col = 'cell_type'

#The first one should be disease state
conditions = ['stim', 'ctrl']

#latent_key = 'X_pca_harmony'
latent_key = 'X_pca'

batch_name = 'batch'
#batch_name = 'replicate'

In [ ]:
adata = sc.read_h5ad(data_dir / "adata.h5ad")

In [ ]:
adata

In [ ]:
#find condition counts in each batch
batch_counts = adata.obs.groupby([batch_name, condition_col]).size().unstack(fill_value=0)
print(batch_counts)

In [ ]:
adata.obs

In [ ]:
adata.X.toarray()[:5, :5]

In [ ]:
# scCST core functions are provided by the installed `sccst` package.
# (This cell previously inlined a copy of the source; importing the package
# instead keeps the example in sync with src/sccst/core.py.)
from sccst import (
    apply_bh_fdr,
    rra_bonferroni_pvalues,
    compute_celltype_local_calipers,
    select_evenly_spaced_anchors,
    process_focal_cell_rra,
    process_cell_type,
    add_rra_results_to_adata,
    make_results_adata,
)

In [ ]:
cell_importances, gene_names, run_info = process_cell_type(adata, conditions = conditions, condition_col=condition_col, batch_col=batch_name, sample_col='sample', n_neighborhoods=nhood_size, cell_type_col=celltype_col, latent_key=latent_key)

In [ ]:
print(f"Processed {len(cell_importances)} cells out of {len(adata.obs.index)}")

In [ ]:
adata = make_results_adata(
    adata=adata,
    cell_importances=cell_importances,
    run_info=run_info,
    prefix="rra",
    subset_to_results=True,
    copy=True,
    gene_level_keys=[
        "gene_score",
        "avg_expr_diff",
        "directional_fdr"],
    obs_level_keys=[
        "cell_divergence_score",
        "n_sig_genes"]
)

In [ ]:
adata

In [ ]:
adata.layers['rra_gene_score']

In [ ]:
#print mean number of significant genes per cell
print(f"Average number of significant genes per cell: {adata.obs['rra_n_sig_genes'].mean():.2f}")

In [ ]:
adata.write(data_dir / f'adata_{nhood_size}_{conditions[0]}_{conditions[1]}.h5ad')